Prepares BED/bedGraph tracks for pyGenomeTracks visualization. Creates per-species track files for intact, disrupted, solo elements and solo ratio.

In [1]:
import pandas as pd
import subprocess

In [6]:
intact_path = "data/intact_elements_dante.tsv"
disrupted_path = "data/partial_elements_dante.tsv"
solo_path = "data/solo_elements.tsv"
ratio_path = "data/solo-other_ratios.tsv"

position_features_dict = {
    "intact": intact_path,
    "disrupted": disrupted_path,
    "solo": solo_path
}
score_features_dict = {
    "solo_ratio": ratio_path
}

In [7]:
sample_order_path = "data/leaf_names.txt"
samples = pd.read_csv(sample_order_path, header=None)[0].tolist()

In [8]:
for sample in samples:
    fai_path = f"data/genomes/{sample}.fna.bgz.fai"
    chromsizes_path = f"data/pygenometracks/{sample}/chrom.sizes"
    subprocess.run(f"mkdir -p data/pygenometracks/{sample}", shell=True)
    subprocess.run(f"cut -f1,2 {fai_path} > {chromsizes_path}", shell=True)

In [ ]:
for feature, path in position_features_dict.items():
    print(f"Processing {feature} features...")
    df = pd.read_csv(path, sep="\t")
    for sample in samples:
        print(f"Sample: {sample}")
        sample_df = df[df["sample"] == sample]
        if sample_df.empty:
            print(f"Sample {sample} not found in {feature} data.")
            continue
        sample_df = sample_df.drop(columns=["sample"])
        sample_df["center"] = (sample_df["start"] + sample_df["end"]) // 2
        bed_path = f"data/pygenometracks/{sample}/{feature}.bed"
        with open(bed_path, "w") as bed_file:
            for _, row in sample_df.iterrows():
                bed_file.write(
                f"{row['chr']}\t{row['center']-1}\t{row['center']}\t1\n"
                )
        chromsizes_path = f"data/pygenometracks/{sample}/chrom.sizes"
        bedgraph_path = f"data/pygenometracks/{sample}/{feature}.bedgraph"
        subprocess.run(
            [
                "bedtools",
                "genomecov",
                "-bga",
                "-i",
                bed_path,
                "-g",
                chromsizes_path
            ],
            stdout=open(bedgraph_path, "w"),
        )
        # sort bedgraph
        subprocess.run(
            [
                "sort",
                "-k1,1",
                "-k2,2n",
                bedgraph_path,
            ],
            stdout=open(bedgraph_path + ".sorted", "w"),
        )
        # create bigwig
        bigwig_path = f"data/pygenometracks/{sample}/{feature}.bw"
        subprocess.run(
            [
                "workflow/scripts/bedGraphToBigWig",
                bedgraph_path + ".sorted",
                chromsizes_path,
                bigwig_path,
            ],
        )
            
        
    

In [ ]:
feature = "solo_ratio"
feature_path = ratio_path
df = pd.read_csv(feature_path, sep="\t")
for sample in samples:
    print(f"Sample: {sample}")
    sample_df = df[df["sample"] == sample]
    if sample_df.empty:
        print(f"Sample {sample} not found in {feature} data.")
        continue
    sample_df = sample_df.drop(columns=["sample"])
    sample_df["center"] = (sample_df["bin_start"] + sample_df["bin_end"]) // 2
    bedgraph_path = f"data/pygenometracks/{sample}/{feature}.bedgraph"
    with open(bedgraph_path, "w") as bed_file:
        for _, row in sample_df.iterrows():
            bed_file.write(
            f"{row['chr']}\t{row['center']-1}\t{row['center']}\t{row['all_solo-other_ratio']}\n"
            )
    chromsizes_path = f"data/pygenometracks/{sample}/chrom.sizes"
    # sort bedgraph
    subprocess.run(
        [
            "sort",
            "-k1,1",
            "-k2,2n",
            bedgraph_path,
        ],
        stdout=open(bedgraph_path + ".sorted", "w"),
    )
    # create bigwig
    bigwig_path = f"data/pygenometracks/{sample}/{feature}.bw"
    subprocess.run(
        [
            "workflow/scripts/bedGraphToBigWig",
            bedgraph_path + ".sorted",
            chromsizes_path,
            bigwig_path,
        ],
    )

# Create plots with pygenometracks
```bash
conda activate pygenometracks
awk 'BEGIN{OFS="\t"} {print $1, 0, $2}' chrom.sizes > regions.bed
make_tracks_file --trackFiles intact.bw disrupted.bw solo.bw -o track.ini
pyGenomeTracks --tracks track.ini --BED regions.bed -o TEs.svg
```
this is a basic track, modified track.ini file for better visualization:

```ini
[x-axis]

[spacer]
height = 0.5

[solo]
file = solo.bw
title = 
height = 2
color = #cf9fddff
min_value = 0
number_of_bins = 200
nans_to_zeros = true
summary_method = sum
show_data_range = true
file_type = bigwig
    

[intact]
file = intact.bw
title = 
height = 2
overlay_previous = share-y
color = #5d7a2fff
min_value = 0
number_of_bins = 200
nans_to_zeros = true
summary_method = sum
show_data_range = true
file_type = bigwig
    
[disrupted]
file = disrupted.bw
title = 
height = 2
overlay_previous = share-y
color = #8a7a3bff
min_value = 0
number_of_bins = 200
nans_to_zeros = true
summary_method = sum
show_data_range = true
file_type = bigwig

[solo_ratio]
file = solo_ratio.bw
title = 
height = 2
overlay_previous = yes

color = #000080
min_value = 0
number_of_bins = 50
nans_to_zeros = false
summary_method = mean
show_data_range = true
file_type = bigwig
```

run on all species:

```bash
cd pygenometracks
for species in */; do
    echo "Processing ${species%/}..."
    cd "$species"
    
    # Create regions.bed
    awk 'BEGIN{OFS="\t"} {print $1, 0, $2}' chrom.sizes > regions.bed
    
    # Generate images
    pyGenomeTracks --tracks ../track.ini --BED regions.bed -o TEs.svg
    
    cd ..
done
```